# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [6]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/emotion_annotated/metadata/pilot_video_emotion_features.csv"
manifest_path = ROOT / "exports/pilot/csv/pilot_manifest_export.csv"
detector_path = ROOT / "datasets/detector_processed/pilot_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


200 200 200


## 2. Merge

In [7]:

df = manifest_df.merge(emotion_df, on="video_id")
df = df.merge(detector_df, on="video_id")

print(df.shape)
df.head()


(200, 83)


,video_id,path,relative_path,split_x,label_x,source_subset,manipulation_family_x,manipulation_type_x,identity,source_identity,...,mean_score_thankfulness_gratitude,mean_score_triumph,label,split,manipulation_family,manipulation_type,n_face_frames_y,video_score_mode,detector_score,detector_pred
0,YouTube-real__00246__bc0e9215,videos/real/YouTube-real/00246.mp4,YouTube-real/00246.mp4,train,real,YouTube-real,NaN,YouTube-real,NaN,NaN,...,0.715369,-1.077735,real,train,NaN,YouTube-real,76,mean,0.270464,0
1,Celeb-real__id19_0003__02a18478,videos/real/Celeb-real/id19_0003.mp4,Celeb-real/id19_0003.mp4,test,real,Celeb-real,NaN,Celeb-real,id19,id19,...,3.180103,0.175943,real,test,NaN,Celeb-real,57,mean,0.435595,0
2,Celeb-real__id5_0001__7446e574,videos/real/Celeb-real/id5_0001.mp4,Celeb-real/id5_0001.mp4,train,real,Celeb-real,NaN,Celeb-real,id5,id5,...,1.921226,-0.706379,real,train,NaN,Celeb-real,64,mean,0.918671,1
3,Celeb-synthesis__FaceSwap__MobileFaceSwap__id2...,videos/fake/FaceSwap/id21_id1_0005.mp4,Celeb-synthesis/FaceSwap/MobileFaceSwap/id21_i...,val,fake,Celeb-synthesis,FaceSwap,MobileFaceSwap,id21,id21,...,2.218336,0.044266,fake,val,FaceSwap,MobileFaceSwap,28,mean,0.665731,1
4,Celeb-synthesis__FaceReenact__MCNET__id35_id32...,videos/fake/FaceReenact/id35_id32_0006.mp4,Celeb-synthesis/FaceReenact/MCNET/id35_id32_00...,train,fake,Celeb-synthesis,FaceReenact,MCNET,id35,id35,...,0.833108,-1.207484,fake,train,FaceReenact,MCNET,28,mean,0.998363,1


## 3. Metrics (baseline)

In [8]:

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"]
y_pred = df["detector_score"]

auc = roc_auc_score(y_true, y_pred)
acc = accuracy_score(y_true, (y_pred > 0.5).astype(int))
f1 = f1_score(y_true, (y_pred > 0.5).astype(int))

print("AUC:", auc)
print("ACC:", acc)
print("F1:", f1)


TypeError: Labels in y_true and y_pred should be of the same type. Got y_true=['fake' 'real'] and y_pred=[0 1]. Make sure that the predictions provided by the classifier coincides with the true labels.

## 4. Emotion analysis

In [9]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

for g in df["arousal_bin"].unique():
    sub = df[df["arousal_bin"] == g]
    auc = roc_auc_score(sub["label"], sub["detector_score"])
    print(g, "AUC:", auc, "n=", len(sub))


mid AUC: 0.41500474833808165 n= 66
low AUC: 0.46574074074074073 n= 67
high AUC: 0.5623885918003565 n= 67


## 5. Fusion model

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]

X = df[features].fillna(0)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score
print("Fusion AUC:", roc_auc_score(y_test, y_pred))


## 6. Compare

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(df["label"], df["detector_score"])

emotion_features = df[[
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]].fillna(0)

model2 = LogisticRegression(max_iter=1000)
model2.fit(emotion_features, df["label"])
emotion_pred = model2.predict_proba(emotion_features)[:, 1]

emotion_auc = roc_auc_score(df["label"], emotion_pred)

print("Baseline AUC:", baseline_auc)
print("Emotion AUC:", emotion_auc)
